In [1]:
import os
import sys
from pathlib import Path
from __future__ import annotations


import matplotlib.pyplot as plt
import numpy as np

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from my_nn import (
    MLP,
    accuracy,
    load_mnist,
    train_epoch_sgd,
    build_optimizer_state,
)

In [2]:
def run_training(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    init_method: str,
    dropout_p: float,
    l2_lambda: float,
    optimizer: str,
    lr: float,
    epochs: int,
    batch_size: int,
    seed: int = 42,
) -> list[float]:
    model = MLP(
        input_dim=784,
        hidden_dims=[96, 64],
        num_classes=10,
        init_method=init_method,
        dropout_p=dropout_p,
        seed=seed,
    )
    opt_state = build_optimizer_state(model, optimizer)
    test_accs: list[float] = []
    for epoch in range(epochs):
        tl, ta = train_epoch_sgd(
            model,
            x_train,
            y_train,
            batch_size=batch_size,
            lr=lr,
            l2_lambda=l2_lambda,
            optimizer=optimizer,
            opt_state=opt_state,
        )
        logits_te = model.forward(x_test, train=False)
        te_acc = accuracy(logits_te, y_test)
        test_accs.append(te_acc)
        if epoch % 5 == 0 or epoch == 1:
            print(
                f"epoch {epoch:3d}  train_loss={tl:.4f}  train_acc={ta:.4f}  test_acc={te_acc:.4f}"
            )
    return test_accs

In [4]:
x_train, y_train, x_test, y_test = load_mnist()
epochs = 30
batch_size = 32

In [ ]:
optim_specs = [
    ("RMSprop", "he", 0.0, 0.0, "rmsprop", 0.001),
    ("Adam", "he", 0.0, 0.0, "adam", 0.002),
]
curves_opt: list[tuple[str, list[float]]] = []
for label, init_m, do, l2, opt, lr in optim_specs:
    acc = run_training(
        x_train,
        y_train,
        x_test,
        y_test,
        init_m,
        do,
        l2,
        opt,
        lr,
        epochs,
        batch_size,
        seed=44,
    )
    curves_opt.append((label, acc))

epoch   0  train_loss=0.2367  train_acc=0.9552  test_acc=0.9527
epoch   1  train_loss=0.1140  train_acc=0.9700  test_acc=0.9640
